# LLM4Rec: Mode A vs Mode C Evaluation
**Letter-based log-prob ranking** — compares text candidates (Mode A) vs soft-token candidates (Mode C).

**Prerequisites** (already trained):
- `checkpoints/projected_embeddings.pt` — cached adapter outputs for all movies
- `checkpoints/id_maps.json` — MovieLens ID ↔ index mapping
- `checkpoints/adapter.pt` — trained MLP adapter
- `test_ranking_prompts.json` + `test_ranking.csv` — eval data

**Step order:**
1. Mount Drive & set paths
2. Install dependencies
3. Verify checkpoints exist
4. Run smoke test (10 examples)
5. Run full evaluation (200 examples)
6. View results

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# IMPORTANT: Change this path to where your ECS172 folder is in Google Drive
PROJECT_DIR = '/content/drive/MyDrive/ECS172'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())
!ls

## Cell 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Done.')

## Cell 3: Verify Checkpoints Exist
This cell checks all required files before loading anything heavy.

In [ ]:
from pathlib import Path
import json
import torch

required = [
    'checkpoints/projected_embeddings.pt',
    'checkpoints/id_maps.json',
    'checkpoints/adapter.pt',
    'test_ranking_prompts.json',
    'test_ranking.csv',
]
for f in required:
    p = Path(f)
    if p.exists():
        size = p.stat().st_size / 1024
        print(f'  ✓ {f} ({size:.0f} KB)')
    else:
        print(f'  ✗ {f} — MISSING!')

# Quick shape check
proj = torch.load('checkpoints/projected_embeddings.pt', map_location='cpu', weights_only=True)
id_maps = json.loads(Path('checkpoints/id_maps.json').read_text())
n_items = id_maps['n_items']
print(f'\n  projected_embeddings shape: {tuple(proj.shape)}')
print(f'  n_items in id_maps: {n_items}')
assert proj.shape[0] == n_items, f'Shape mismatch: {proj.shape[0]} vs {n_items}'
print('  ✓ Shape check passed')

## Cell 4: Configuration
Set these 3 variables before running. Everything else is automatic.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIG — edit these
# ═══════════════════════════════════════════════════════════════════

# LLM to use. Start with the base model for validation,
# then switch to your fine-tuned LoRA for the headline comparison.
MODEL = 'unsloth/Llama-3.2-1B-Instruct'
# MODEL = './llama31-1b-movielens-full-final'

# How many test examples to evaluate (None = all)
MAX_SAMPLES = 200  # Set to 10 for quick smoke test

# Chat template: start False (raw prompts), enable after validation
USE_CHAT_TEMPLATE = False

## Cell 5: Run Evaluation
Runs Mode A and Mode C side by side using `scripts/eval_ac.py`.

In [ ]:
cmd = f'python scripts/eval_ac.py --model {MODEL} --max-samples {MAX_SAMPLES} --modes A,C'
if USE_CHAT_TEMPLATE:
    cmd += ' --chat-template'
print(f'Running: {cmd}\n')
!{cmd}

## Cell 6: View Detailed Results
Reads `results_ac.json` and shows the comparison table plus per-example predictions.

In [ ]:
import json
from pathlib import Path

results = json.loads(Path('results_ac.json').read_text())

# ── Metrics table ──
ks = [1, 3, 5, 10]
header = f"{'Mode':<25}"
for k in ks:
    header += f"  {'HR@'+str(k):>6}"
for k in ks:
    header += f"  {'NDCG@'+str(k):>8}"
print(header)
print('─' * len(header))

for label, m in results['metrics'].items():
    row = f"{label:<25}"
    for k in ks:
        row += f"  {m.get(f'HR@{k}', 0):>6.3f}"
    for k in ks:
        row += f"  {m.get(f'NDCG@{k}', 0):>8.4f}"
    print(row)

# ── Sample predictions ──
print('\n── Sample Predictions ──')
for mode_name, preds in results.get('predictions', {}).items():
    print(f'\n{mode_name}:')
    for ex in preds[:5]:
        status = '✓' if ex['correct_top1'] else '✗'
        probs_top3 = sorted(ex['probs'].items(), key=lambda x: -x[1])[:3]
        probs_str = ', '.join(f'{k}={v:.3f}' for k, v in probs_top3)
        print(f"  user {ex['user_id']}: pred={ex['top1_letter']} true={ex['true_letter']} "
              f"rank={ex['true_rank']} {status}  top3: [{probs_str}]")

## Cell 7: Probability Distribution Analysis
Check if the model is actually using the soft tokens or producing near-uniform distributions.

In [ ]:
import numpy as np

for mode_name, preds in results.get('predictions', {}).items():
    all_probs = [list(ex['probs'].values()) for ex in preds]
    max_probs = [max(p) for p in all_probs]
    entropies = [-sum(pi * np.log(pi + 1e-10) for pi in p) for p in all_probs]
    
    n_cands = len(all_probs[0]) if all_probs else 10
    uniform_entropy = -n_cands * (1/n_cands) * np.log(1/n_cands)
    
    print(f'\n{mode_name}:')
    print(f'  Max prob:  mean={np.mean(max_probs):.3f}  std={np.std(max_probs):.3f}  '
          f'min={np.min(max_probs):.3f}  max={np.max(max_probs):.3f}')
    print(f'  Entropy:   mean={np.mean(entropies):.3f}  (uniform={uniform_entropy:.3f})')
    
    # Check: near-uniform means model isn't using signals
    if np.mean(max_probs) < 0.15:
        print(f'  ⚠️  Near-uniform distribution — model may not be using the candidates effectively')
    else:
        print(f'  ✓ Peaked distribution — model is discriminating between candidates')

## Cell 8: Option 1 — Chat Template Experiment
Tests if the raw `Answer:` prompt is causing the positional bias in Mode C. Wraps the prompt in the LLM's native instruction template.

In [ ]:
# Run the evaluation with --chat-template enabled
# We run on a 50-sample subset to quickly see if the position-A bias breaks
cmd_chat = f"python scripts/eval_ac.py --model {MODEL} --max-samples 50 --modes A,C --chat-template"
print(f"Running: {cmd_chat}\n")
!{cmd_chat}